In [1]:
import scanpy as sc
import tempfile
import requests

def load_from_url(url):
    response = requests.get(url)
    with tempfile.NamedTemporaryFile(suffix=".h5ad", delete=False) as tmp_file:
        tmp_file.write(response.content)
        tmp_file.flush()
        return sc.read_h5ad(tmp_file.name)

# Load AD and cerebellum data
ad_url = "https://datasets.cellxgene.cziscience.com/42f1ec32-aaa2-4f08-8f78-a34f9b993ea8.h5ad"
cereb_url = "https://datasets.cellxgene.cziscience.com/b07a1ded-bf24-48ef-b32b-a58de97b8080.h5ad"

adata = load_from_url(ad_url)
cereb_data = load_from_url(cereb_url)

In [2]:
shared_genes = list(set(adata.var_names) & set(cereb_data.var_names))
adata_sub = adata[:, shared_genes]
cereb_sub = cereb_data[:, shared_genes]

In [3]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

X_ad = adata_sub.X
y_ad = LabelEncoder().fit_transform(adata_sub.obs["disease"])  # e.g., AD vs Control

X_cereb = cereb_sub.X
if "development_stage" in cereb_sub.obs.columns:
    y_cereb = LabelEncoder().fit_transform(cereb_sub.obs["development_stage"])
else:
    y_cereb = np.zeros(X_cereb.shape[0])


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectFromModel

# SVM for AD
svm_ad = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_ad.fit(X_ad, y_ad)
selected_ad = np.abs(svm_ad.coef_).sum(axis=0)

# SVM for Cerebellum
svm_cereb = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_cereb.fit(X_cereb, y_cereb)
selected_cereb = np.abs(svm_cereb.coef_).sum(axis=0)

# Top genes
top_n = 50
genes_array = np.array(shared_genes)
top_ad_genes = genes_array[np.argsort(selected_ad)[-top_n:]]
top_cereb_genes = genes_array[np.argsort(selected_cereb)[-top_n:]]
overlap = set(top_ad_genes) & set(top_cereb_genes)
print(f"Overlap ({len(overlap)}): {overlap}")


In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# Combine inputs
X_combined = np.vstack([X_ad, X_cereb])
y_ad_combined = np.concatenate([y_ad, [-1]*X_cereb.shape[0]])
y_cereb_combined = np.concatenate([[-1]*X_ad.shape[0], y_cereb])

# Train separate MLPs (simplified multitask)
mlp_ad = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
mlp_ad.fit(X_ad, y_ad)
print("AD classification report:")
print(classification_report(y_ad, mlp_ad.predict(X_ad)))

mlp_cereb = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
mlp_cereb.fit(X_cereb, y_cereb)
print("Cerebellum classification report:")
print(classification_report(y_cereb, mlp_cereb.predict(X_cereb)))


In [ ]:
from sklearn.inspection import permutation_importance

perm_ad = permutation_importance(mlp_ad, X_ad, y_ad, n_repeats=10, random_state=42)
top_nn_ad = genes_array[np.argsort(perm_ad.importances_mean)[-top_n:]]

perm_cereb = permutation_importance(mlp_cereb, X_cereb, y_cereb, n_repeats=10, random_state=42)
top_nn_cereb = genes_array[np.argsort(perm_cereb.importances_mean)[-top_n:]]

overlap_nn = set(top_nn_ad) & set(top_nn_cereb)
print(f"Overlap in NN-important genes: {overlap_nn}")
